In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Colon Cancer and Its Risk Factors

This notebook presents the prepared datasets from `Analysis.ipynb` as an interactive dashboard. Run the analysis notebook first to refresh its CSV inputs.

In [2]:
# Dashboard heading and chart containers
title = widgets.HTML("<h1>Colon Cancer and Its Risk Factors</h1>")
subtitle = widgets.HTML("<h3>An analytical overview of key risk factors and trends</h3>")
visuals = [widgets.Output() for _ in range(6)]

# 1. Colon cancer incidence by location and age group
with visuals[0]:
    display(widgets.HTML("<h2>1. Colon Cancer Incidence by Location</h2>"))
    incidence = pd.read_csv("incidence_by_location.csv")
    locations = sorted(incidence["Location"].dropna().unique())
    age_groups = [column for column in incidence.columns if column[:1].isdigit()]

    figure = go.Figure()
    for location in locations:
        location_rows = incidence.loc[incidence["Location"].eq(location), age_groups]
        figure.add_bar(
            x=age_groups,
            y=location_rows.sum().to_numpy(),
            name=location,
            visible=(location == locations[0]),
            hovertemplate=(
                f"<b>Location:</b> {location}<br>"
                "<b>Age group:</b> %{x}<br>"
                "<b>Cases:</b> %{y}<extra></extra>"
            ),
        )

    figure.update_layout(
        title="Colon Cancer Incidence by Age Group and Location",
        xaxis_title="Age group",
        yaxis_title="Number of cases",
        barmode="group",
        updatemenus=[{
            "buttons": [
                {
                    "label": location,
                    "method": "update",
                    "args": [{"visible": [candidate == location for candidate in locations]}],
                }
                for location in locations
            ],
            "direction": "down",
            "showactive": True,
            "x": 0.5,
            "xanchor": "center",
            "y": 1.15,
        }],
        hoverlabel={"bgcolor": "white", "font_size": 12},
    )
    figure.show()
    fig1 = figure

# 2. Tobacco use and colon cancer incidence
with visuals[1]:
    display(widgets.HTML("<h2>2. Tobacco Use and Colon Cancer</h2>"))
    tobacco_incidence = pd.read_csv("long_data.csv")
    figure = px.scatter(
        tobacco_incidence,
        x="Numeric",
        y="Cancer_Incidence",
        color="Gender",
        hover_data=["Country", "Age_Group"],
        title="Tobacco Use vs. Colon Cancer Incidence",
        labels={
            "Numeric": "Tobacco use (% of population)",
            "Cancer_Incidence": "Colon cancer cases",
        },
    )
    figure.show()
    fig2 = figure

# 3. Colon cancer incidence by continent, gender, and age group
with visuals[2]:
    display(widgets.HTML("<h2>3. Colon Cancer Incidence by Continent</h2>"))
    continent_data = pd.read_csv("melted.csv")
    figure = go.Figure(
        go.Heatmap(
            x=continent_data["Age_Group"],
            y=continent_data["Continent"] + " - " + continent_data["Gender"],
            z=continent_data["Cancer_Incidence"],
            customdata=continent_data["Numeric"],
            colorscale="Blues",
            colorbar={"title": "Cancer incidence"},
            hovertemplate=(
                "<b>Continent and gender:</b> %{y}<br>"
                "<b>Age group:</b> %{x}<br>"
                "<b>Cancer incidence:</b> %{z}<br>"
                "<b>Average tobacco use:</b> %{customdata}<extra></extra>"
            ),
        )
    )
    figure.update_layout(
        title="Colon Cancer Incidence by Age Group and Continent",
        xaxis_title="Age group",
        yaxis_title="Continent and gender",
    )
    figure.show()
    fig3 = figure

# 4. Country-level salt intake
with visuals[3]:
    display(widgets.HTML("<h2>4. Salt Intake by Country</h2>"))
    salt_data = pd.read_csv("salt_by_country.csv")
    figure = px.choropleth(
        salt_data,
        locations="Country",
        locationmode="country names",
        color="Salt_Intake",
        hover_name="Country",
        color_continuous_scale="YlOrBr",
        title="Mean Population Salt Intake (g/day)",
    )
    figure.update_layout(width=900, height=800)
    figure.show()
    fig4 = figure

# 5. Country-level colon cancer incidence
with visuals[4]:
    display(widgets.HTML("<h2>5. Colon Cancer Incidence by Country</h2>"))
    colon_data = pd.read_csv("colon_by_country.csv")
    figure = px.choropleth(
        colon_data,
        locations="Country",
        locationmode="country names",
        color="Colon_Cancer_Rate",
        hover_name="Country",
        color_continuous_scale="YlOrRd",
        title="Colon Cancer Cases by Country",
    )
    figure.update_layout(width=900, height=800)
    figure.show()
    fig5 = figure

# 6. Correlations with other risk factors
with visuals[5]:
    display(widgets.HTML("<h2>6. Colon Cancer and Other Risk Factors</h2>"))
    risk_data = pd.read_csv("merged_data.csv")
    risk_factors = [
        "Cancer Incidence",
        "Alcohol_Consumption",
        "Obesity_Prevalence",
        "Physical_Inactivity",
    ]
    correlation = risk_data[risk_factors].corr()
    figure = px.imshow(
        correlation,
        text_auto=".2f",
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        title="Colon Cancer Incidence and Other Risk Factors",
    )
    figure.show()
    fig6 = figure

# Arrange the charts in a single dashboard.
dashboard = widgets.VBox([
    widgets.VBox([title, subtitle, visuals[0]]),
    visuals[1],
    visuals[2],
    widgets.HBox([visuals[3], visuals[4]]),
    visuals[5],
])
display(dashboard)